In [1]:
import sys
from pathlib import Path

# プロジェクトルート
project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")

# Python がソースコードを探す場所として src/ を追加
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("sys.path に追加された src:", src_path)

# ✅ ここがポイント！
# src を sys.path に入れたので、「ifc_ingestor」がトップレベルパッケージになる
# → import するときは「from ifc_ingestor...」で OK（"src." は付けない）
from ifc_ingestor.ingestor import load_ifc_bundle

print("IMPORT SUCCESS:", load_ifc_bundle)

sys.path に追加された src: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\src
IMPORT SUCCESS: <function load_ifc_bundle at 0x000001E06DCED580>


In [2]:
# --- IFC の読み込み ---
ifc_path = project_root / "data" / "ifc" / "raw" / "51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc"
bundle = load_ifc_bundle(str(ifc_path))

print("\n=== IFC Loaded ===")
print("schema:", bundle.schema)
print("model type:", type(bundle.model))


# --- BDNS タグ抽出 ---
from bdns_extractor.extractor import extract_bdns_tags

bdns_assets = extract_bdns_tags(bundle)

print("\n=== BDNS Extraction Result ===")
print("BDNS-tagged assets:", len(bdns_assets.items))

# 最初の数件を表示
for asset in bdns_assets.items[:10]:
    print(
        "GUID:", asset.ifc_guid,
        "| BDNS:", asset.bdns_tag,
        "| Name:", asset.name,
        "| Class:", asset.raw_ifc_class
    )


=== IFC Loaded ===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>

=== BDNS Extraction Result ===
BDNS-tagged assets: 6511
GUID: 2HwPth0ED3muP_ET7DKBaz | BDNS: outlet - double switched socket outlet | Name: DSSO-00001 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK99g | BDNS: outlet - double switched socket outlet | Name: DSSO-00002 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9CM | BDNS: outlet - double switched socket outlet | Name: DSSO-00003 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9Ws | BDNS: outlet - double switched socket outlet | Name: DSSO-00004 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9_L | BDNS: outlet - double switched socket outlet | Name: DSSO-00005 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7lq | BDNS: outlet - double switched socket outlet | Name: DSSO-00006 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7qD | BDNS: outlet - double switched socket outlet | Name: DSSO-00007 | Class: IfcFlowTerminal
GUID: 3ID7pTUZ96xeg742M

In [3]:
# 追加
spatial = extract_spatial_elements(bundle)

# 別出力として JSON 化する例
import json
from bdns_extractor.extractor import extract_spatial_elements
with open("spatial_elements.json", "w", encoding="utf-8") as f:
    json.dump(
        [e.__dict__ for e in spatial.items],
        f,
        ensure_ascii=False,
        indent=2
    )

NameError: name 'extract_spatial_elements' is not defined

In [3]:
from pathlib import Path
from class_mapper.mapper import map_bdns_to_brick_equipment

project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")
mapping_csv_path = project_root / "resources" / "bdns" / "bdns_assets_to_brick_map.csv"

base_ns = "http://example.com/brick#"

equip_set = map_bdns_to_brick_equipment(
    tagged_assets=bdns_assets,
    base_ns=base_ns,
    crosswalk_equip=None,
    mapping_csv_path=str(mapping_csv_path),
)

total = len(equip_set.items)
unmapped = sum(1 for e in equip_set.items if e.extra_props.get("unmapped") is True)
mapped = total - unmapped

print("=== Mapping summary ===")
print("total,", total)
print("mapped,", mapped)
print("unmapped,", unmapped)
print("ratio_unmapped,", f"{(unmapped/total*100):.2f}%" if total else "n/a")

print("\n=== examples ===")
dssos = [e for e in equip_set.items if isinstance(e.bdns_tag, str) and str(e.bdns_tag).upper().startswith("DSSO")]
for e in dssos[:10]:
    print(
        "GUID:", e.ifc_guid,
        "| brick_class:", e.brick_class,
        "| label:", e.label,
        "| mapping_source:", e.extra_props.get("mapping_source"),
        "| unmapped:", e.extra_props.get("unmapped"),
    )
print("count,", len(dssos))

=== Mapping summary ===
total, 6511
mapped, 6511
unmapped, 0
ratio_unmapped, 0.00%

=== examples ===
GUID: 2HwPth0ED3muP_ET7DKBaz | brick_class: brick:Electrical_Outlet | label: DSSO-00001 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK99g | brick_class: brick:Electrical_Outlet | label: DSSO-00002 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK9CM | brick_class: brick:Electrical_Outlet | label: DSSO-00003 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK9Ws | brick_class: brick:Electrical_Outlet | label: DSSO-00004 | mapping_source: code | unmapped: None
GUID: 2HwPth0ED3muP_ET7DK9_L | brick_class: brick:Electrical_Outlet | label: DSSO-00005 | mapping_source: code | unmapped: None
GUID: 1zhYdOoOz9rBBDvGWES7lq | brick_class: brick:Electrical_Outlet | label: DSSO-00006 | mapping_source: code | unmapped: None
GUID: 1zhYdOoOz9rBBDvGWES7qD | brick_class: brick:Electrical_Outlet | label: DSSO-00007 | mapping_source: code | unmapped: None
GUI

In [4]:
equip_set.items

[BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DKBaz', brick_class='brick:Electrical_Outlet', label='DSSO-00001', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched socket outlet', 'mapping_source': 'code'}),
 BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DK99g', brick_class='brick:Electrical_Outlet', label='DSSO-00002', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched socket outlet', 'mapping_source': 'code'}),
 BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DK9CM', brick_class='brick:Electrical_Outlet', label='DSSO-00003', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched socket outlet', 'mapping_source': 'code'}),
 BrickEquipment(ifc_guid='2HwPth0ED3muP_ET7DK9Ws', brick_class='brick:Electrical_Outlet', label='DSSO-00004', bdns_tag='DSSO', extra_props={'raw_ifc_class': 'IfcFlowTerminal', 'bdns_label': 'outlet - double switched sock

In [5]:

#集計
total = len(equip_set.items)
unmapped = sum(1 for e in equip_set.items if e.extra_props.get("unmapped") is True)
mapped = total - unmapped
unmapped_ratio = unmapped / total if total else 0.0

print("=== Summary ===")
print(f"total: {total}")
print(f"mapped: {mapped}")
print(f"unmapped: {unmapped}")
print(f"unmapped ratio: {unmapped_ratio:.2%}")

# 代表サンプル表示
print("\n=== Sample mapped items (up to 10) ===")
count = 0
for e in equip_set.items:
    if e.extra_props.get("unmapped") is True:
        continue
    print(
        "GUID:", e.ifc_guid,
        "| brick_class:", e.brick_class,
        "| label:", e.label,
        "| bdns_tag:", e.bdns_tag,
        "| raw_ifc_class:", e.extra_props.get("raw_ifc_class"),
        "| source:", e.extra_props.get("mapping_source"),
    )
    count += 1
    if count >= 10:
        break

print("\n=== Sample unmapped items (up to 10) ===")
count = 0
for e in equip_set.items:
    if e.extra_props.get("unmapped") is not True:
        continue
    print(
        "GUID:", e.ifc_guid,
        "| fallback_class:", e.brick_class,
        "| label:", e.label,
        "| bdns:", e.bdns_tag,
        "| raw_ifc_class:", e.extra_props.get("raw_ifc_class"),
    )
    count += 1
    if count >= 10:
        break


=== Summary ===
total: 6511
mapped: 6511
unmapped: 0
unmapped ratio: 0.00%

=== Sample mapped items (up to 10) ===
GUID: 2HwPth0ED3muP_ET7DKBaz | brick_class: brick:Electrical_Outlet | label: DSSO-00001 | bdns_tag: DSSO | raw_ifc_class: IfcFlowTerminal | source: code
GUID: 2HwPth0ED3muP_ET7DK99g | brick_class: brick:Electrical_Outlet | label: DSSO-00002 | bdns_tag: DSSO | raw_ifc_class: IfcFlowTerminal | source: code
GUID: 2HwPth0ED3muP_ET7DK9CM | brick_class: brick:Electrical_Outlet | label: DSSO-00003 | bdns_tag: DSSO | raw_ifc_class: IfcFlowTerminal | source: code
GUID: 2HwPth0ED3muP_ET7DK9Ws | brick_class: brick:Electrical_Outlet | label: DSSO-00004 | bdns_tag: DSSO | raw_ifc_class: IfcFlowTerminal | source: code
GUID: 2HwPth0ED3muP_ET7DK9_L | brick_class: brick:Electrical_Outlet | label: DSSO-00005 | bdns_tag: DSSO | raw_ifc_class: IfcFlowTerminal | source: code
GUID: 1zhYdOoOz9rBBDvGWES7lq | brick_class: brick:Electrical_Outlet | label: DSSO-00006 | bdns_tag: DSSO | raw_ifc_class

In [6]:
# カーネルが参照している Python 実行環境の確認と src 追加
import sys, pkgutil, importlib
from pathlib import Path

project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("Python:", sys.executable)
print("src added:", src_path)

# rdflib の存在確認と必要ならインストール
if pkgutil.find_loader("rdflib") is None:
    # カーネル環境にインストール
    %pip install rdflib

import rdflib
print("rdflib version:", rdflib.__version__)

Python: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\.venv\Scripts\python.exe
src added: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\src
rdflib version: 7.6.0


C:\Users\Owner\AppData\Local\Temp\ipykernel_29428\3591875565.py:14: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader("rdflib") is None:


In [7]:
from rdf_writer.writer import write_equipments_turtle

base_ns = "https://id.morgate51.org/project#"
write_equipments_turtle(equip_set, base_ns, r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\data\outputmorgate51_equipment.ttl")